<a href="https://colab.research.google.com/github/fourmodern/2025_aidrugdiscovery/blob/main/Day4/reinvent4_denovo_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REINVENT4 — de novo 분자 생성 + RL 최적화 (QED)

**AI 신약설계 실습 · Day4 (심화)**

[REINVENT4](https://github.com/MolecularAI/REINVENT4) (AstraZeneca, Loeffler et al., *J Cheminform* 16:20, 2024)는
산업계에서 널리 쓰이는 **오픈소스 분자 생성·최적화 프레임워크**입니다. 이 노트북에서는:

1. **de novo 샘플링** — 사전학습 prior로 새로운 SMILES 생성
2. **생성물 평가** — 유효성(validity)·고유성(uniqueness)·신규성(novelty)
3. **물성/약물성 필터** — QED·SA·Lipinski Ro5
4. **(심화) 강화학습(staged learning)** — **QED를 목적함수**로 생성을 최적화

> ⚠️ **정직한 한계**
> - 생성 결과는 **후보 제안(가설)** 이며, 합성·assay 실험 검증 전까지 결론이 아닙니다.
> - 여기서 계산하는 수치(QED·SA·MW 등)는 모두 **RDKit 실계산값**입니다(임의 수치 없음).
> - 표적 특이적(예: PDE5) 최적화 하네스는 별도 저장소(`2026_aidrugdiscovery/Day06_LLM_Agent/pde5_harness`)에 있습니다. 이 노트북은 **범용 생성모델 실습**입니다.

> 🖥️ **실행 환경**: **GPU 런타임 필수**(런타임 → 런타임 유형 변경 → GPU). 설치에 수 분 소요되며, 설치 직후 **런타임 재시작**이 한 번 필요할 수 있습니다.

## 0. 런타임 확인 (GPU)

In [ ]:
import sys
print("Python:", sys.version.split()[0])
try:
    import torch
    print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except Exception as e:
    print("torch 미확인:", e)

# GPU 확인 (없으면 CPU로도 sampling은 가능하나 RL 최적화는 매우 느림)
!nvidia-smi -L 2>/dev/null || echo "GPU 없음 → 런타임 유형을 GPU로 변경 권장"


## 1. REINVENT4 설치

REINVENT4는 **pip 패키지가 없어** 저장소를 clone한 뒤 `install.py`로 설치합니다.
Colab(예: Python 3.12 / torch cu128)에서는 `cu126` 옵션을 사용합니다 — cu126 휠은 최신 CUDA 드라이버에서 정상 동작합니다.

> ⏳ 설치는 수 분 걸립니다. **설치 중 torch가 cu126 버전으로 교체될 수 있으며**, 이 경우 아래 import 검증이 실패하면 **런타임 재시작(런타임 → 세션 다시 시작) 후 이 셀 이후부터 다시 실행**하세요.

In [ ]:
%%bash
set -e
if [ ! -d REINVENT4 ]; then
  git clone --depth 1 https://github.com/MolecularAI/REINVENT4.git
fi
cd REINVENT4
# GPU(Colab) → cu126.  CPU만 있으면 아래 줄을 'python install.py cpu' 로 바꾸세요.
python install.py cu126
echo "=== 설치 완료 ==="


In [ ]:
# 설치 검증: reinvent CLI가 잡히는지 확인
import subprocess, shutil
exe = shutil.which("reinvent")
print("reinvent 실행파일:", exe)
out = subprocess.run(["reinvent", "--help"], capture_output=True, text=True)
print(out.stdout[:400] if out.returncode == 0 else out.stderr[:400])
assert out.returncode == 0, "reinvent 미설치 → 위 설치 셀 재실행 또는 런타임 재시작 필요"
print("OK — REINVENT4 사용 가능")


## 2. 사전학습 Prior 다운로드

de novo 생성용 prior(`reinvent_pubchem.prior`, 약 23MB)를 [Zenodo](https://doi.org/10.5281/zenodo.15641296)에서 내려받습니다.

In [ ]:
import os, urllib.request
os.makedirs("priors", exist_ok=True)
PRIOR = "priors/reinvent_pubchem.prior"
URL = "https://zenodo.org/records/20701824/files/reinvent_pubchem.prior?download=1"  # 버전 record (개념 record는 파일 URL 리다이렉트 안 됨)
if not os.path.exists(PRIOR) or os.path.getsize(PRIOR) < 1_000_000:
    print("prior 다운로드 중 …")
    urllib.request.urlretrieve(URL, PRIOR)
print("prior:", PRIOR, "|", round(os.path.getsize(PRIOR)/1e6, 1), "MB")


## 3. de novo 샘플링

REINVENT4는 **TOML 설정 파일 + CLI**로 동작합니다. 먼저 sampling 설정을 작성합니다.
(설정 키는 공식 `configs/sampling.toml` 스키마를 따릅니다.)

In [ ]:
sampling_toml = """
run_type = "sampling"
device = "cuda:0"          # GPU 없으면 "cpu"
json_out_config = "_sampling.json"

[parameters]
model_file = "priors/reinvent_pubchem.prior"
output_file = "sampling.csv"
num_smiles = 200            # 생성 개수
unique_molecules = true     # 중복 제거
randomize_smiles = true
sample_strategy = "multinomial"
temperature = 1.0
"""
with open("sampling.toml", "w") as f:
    f.write(sampling_toml)
print(sampling_toml)


In [ ]:
# de novo 샘플링 실행
!reinvent -l sampling.log sampling.toml
print("=== 로그 끝부분 ===")
!tail -n 8 sampling.log


In [ ]:
import pandas as pd
gen = pd.read_csv("sampling.csv")
print("컬럼:", list(gen.columns))
print("생성 행 수:", len(gen))
# SMILES 컬럼 자동 탐지
smi_col = next((c for c in gen.columns if c.lower() in ("smiles", "canonical_smiles", "output_smiles")), gen.columns[-1])
gen_smiles = gen[smi_col].dropna().astype(str).tolist()
print("SMILES 컬럼:", smi_col, "| 예시:")
for s in gen_smiles[:5]:
    print("  ", s)


## 4. 생성물 평가 — 유효성 · 고유성 · 신규성

- **Validity**: RDKit로 파싱되는 비율
- **Uniqueness**: 유효분자 중 canonical SMILES 고유 비율
- **Novelty**: 참조셋(reference)에 없는 비율 (여기서는 대표 PDE5 저해제 등 소규모 참조셋 기준 — 참조셋에 따라 값이 달라짐)

In [ ]:
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

def canon(s):
    m = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(m) if m else None

# 소규모 참조셋(신규성 기준) — 대표 승인 PDE5 저해제 등. 참조셋은 목적에 맞게 교체 가능.
REFERENCE = [
    "CCCc1nn(C)c2c1nc([nH]c2=O)-c1cc(ccc1OCC)S(=O)(=O)N1CCN(C)CC1",  # sildenafil
    "O=C1N(C)CC(=O)N2[C@@H]1Cc1c([nH]c3ccccc13)[C@H]2c1ccc2c(c1)OCO2", # tadalafil
]
ref_canon = {canon(s) for s in REFERENCE if canon(s)}

valid = [c for c in (canon(s) for s in gen_smiles) if c]
uniq = set(valid)
novel = uniq - ref_canon

n = len(gen_smiles)
print(f"생성           : {n}")
print(f"유효(validity) : {len(valid)}  ({len(valid)/max(n,1):.1%})")
print(f"고유(uniqueness): {len(uniq)}  ({len(uniq)/max(len(valid),1):.1%} of valid)")
print(f"신규(novelty)  : {len(novel)}  ({len(novel)/max(len(uniq),1):.1%} of unique, 참조셋 대비)")


## 5. 물성/약물성 필터 — QED · SA · Lipinski Ro5

생성된 유효·고유 분자에 대해 약물유사도(QED), 합성접근성(SA), Lipinski Rule of 5를 **RDKit로 실계산**합니다.

In [ ]:
import os, sys
from rdkit.Chem import Descriptors, QED, Lipinski, Crippen, rdMolDescriptors, Draw

def sa_score(mol):
    try:
        from rdkit.Chem import RDConfig
        sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
        import sascorer
        return round(sascorer.calculateScore(mol), 2)
    except Exception:
        return None

rows = []
for s in sorted(uniq):
    m = Chem.MolFromSmiles(s)
    if not m:
        continue
    mw, logp = Descriptors.MolWt(m), Crippen.MolLogP(m)
    hbd, hba = Lipinski.NumHDonors(m), Lipinski.NumHAcceptors(m)
    q, sa = QED.qed(m), sa_score(m)
    ro5 = sum([mw <= 500, logp <= 5, hbd <= 5, hba <= 10])
    rows.append({"smiles": s, "MW": round(mw,1), "logP": round(logp,2),
                 "HBD": hbd, "HBA": hba, "QED": round(q,3), "SA": sa,
                 "Ro5_ok": ro5 >= 3,
                 "pass": (ro5 >= 3) and (q >= 0.5) and (sa is None or sa <= 6.0)})

df = pd.DataFrame(rows).sort_values("QED", ascending=False).reset_index(drop=True)
print("게이트(Ro5≥3 · QED≥0.5 · SA≤6) 통과:", int(df["pass"].sum()), "/", len(df))
df.head(12)


In [ ]:
# 상위 QED 후보 그리드 시각화
top = df[df["pass"]].head(12) if df["pass"].any() else df.head(12)
mols = [Chem.MolFromSmiles(s) for s in top["smiles"]]
legends = [f"QED={q}" for q in top["QED"]]
Draw.MolsToGridImage(mols, legends=legends, molsPerRow=4, subImgSize=(240, 200))


## 6. (심화) 강화학습 — QED 목적함수 최적화 (staged learning)

REINVENT4의 핵심 기능은 **생성을 목적함수(스코어)를 향해 강화학습**하는 것입니다.
여기서는 **QED를 최대화**하도록(+ 분자량 범위·구조 경보 필터) staged learning을 실행합니다.
(스코어링 컴포넌트 이름·구조는 공식 `configs/staged_learning.toml`·`stage2_scoring.toml` 스키마를 따릅니다.)

> ⏳ GPU에서도 수십 step 학습에 시간이 걸립니다. 데모는 `max_steps`를 작게 잡았습니다.

In [ ]:
staged_toml = """
run_type = "staged_learning"
device = "cuda:0"
tb_logdir = "tb_logs"
json_out_config = "_staged_learning.json"

[parameters]
summary_csv_prefix = "staged_learning"
use_checkpoint = false
prior_file = "priors/reinvent_pubchem.prior"
agent_file = "priors/reinvent_pubchem.prior"
batch_size = 64

[learning_strategy]
type = "dap"
sigma = 128
rate = 0.0001

[[stage]]
chkpt_file = "qed_stage.chkpt"
termination = "simple"
max_score = 0.7
min_steps = 10
max_steps = 60          # 데모용 — 실전은 더 크게

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.QED]
[[stage.scoring.component.QED.endpoint]]
name = "QED Score"
weight = 1.0

[[stage.scoring.component]]
[stage.scoring.component.MolecularWeight]
[[stage.scoring.component.MolecularWeight.endpoint]]
name = "Molecular weight"
weight = 0.5
transform.type = "double_sigmoid"
transform.high = 500.0
transform.low = 200.0
transform.coef_div = 500.0
transform.coef_si = 20.0
transform.coef_se = 20.0
"""
with open("staged_learning.toml", "w") as f:
    f.write(staged_toml)
print(staged_toml)


In [ ]:
# 강화학습 실행 (시간 소요) — GPU 권장
!reinvent -l staged.log staged_learning.toml
print("=== 로그 끝부분 ===")
!tail -n 12 staged.log


In [ ]:
# 학습 결과 로드 — QED 분포가 개선되었는지 확인
import glob, matplotlib.pyplot as plt
csvs = sorted(glob.glob("staged_learning*.csv"))
print("결과 CSV:", csvs)
opt = pd.read_csv(csvs[-1]) if csvs else pd.DataFrame()
print("컬럼:", list(opt.columns)[:12])

# QED 컬럼 탐지 후 최적화 전(de novo) vs 후 비교
qcol = next((c for c in opt.columns if "qed" in c.lower()), None)
if qcol is not None and len(opt):
    fig, ax = plt.subplots(figsize=(7,4))
    ax.hist(df["QED"], bins=20, alpha=0.5, label="de novo (Step 3)", density=True)
    ax.hist(pd.to_numeric(opt[qcol], errors="coerce").dropna(), bins=20, alpha=0.5,
            label="RL 최적화 후", density=True)
    ax.set_xlabel("QED"); ax.set_ylabel("density"); ax.legend()
    ax.set_title("QED 분포: de novo vs RL 최적화")
    plt.tight_layout(); plt.show()
    print("de novo 평균 QED :", round(df["QED"].mean(), 3))
    print("RL 후    평균 QED :", round(pd.to_numeric(opt[qcol], errors="coerce").mean(), 3))
else:
    print("QED 컬럼을 찾지 못했습니다 — opt.columns를 확인해 컬럼명을 맞춰주세요.")


## 정리

- **de novo 샘플링** → **평가(validity·uniqueness·novelty)** → **약물성 필터(QED·SA·Ro5)** → **RL 최적화(QED 목적)** 의 전체 파이프라인을 REINVENT4로 수행했습니다.
- 모든 물성 수치는 **RDKit 실계산값**이며, 생성 분자는 **후보 가설**입니다 — 합성·생물학적 검증이 뒤따라야 합니다.
- 표적 특이적(예: **PDE5 저해제**) 스코어링·검증 게이트·보고서 자동화는 별도 하네스
  (`2026_aidrugdiscovery/Day06_LLM_Agent/pde5_harness`)에서 다룹니다.

**참고문헌**
- Loeffler, H.H. et al. *Reinvent 4: Modern AI-driven generative molecule design.* J Cheminform 16, 20 (2024). https://doi.org/10.1186/s13321-024-00812-5
- REINVENT4 GitHub: https://github.com/MolecularAI/REINVENT4
- Priors (Zenodo): https://doi.org/10.5281/zenodo.15641296